# Project 4 Weather Routing API

### Project Description
An automated Meteorological Routing API Pipeline. It interfaces with Open-Meteo to extract 10-hour forecasts and uses NumPy linear interpolation to calculate weather along a geographic route.

In [1]:
import requests
import pandas as pd
import numpy as np

def get_coordinates(city_name: str):
    res = requests.get("https://geocoding-api.open-meteo.com/v1/search", 
                       params={"name": city_name, "count": 1, "format": "json"}).json()
    return res["results"][0]["latitude"], res["results"][0]["longitude"] if "results" in res else None

def get_10hr_forecast(lat: float, lon: float):
    res = requests.get("https://api.open-meteo.com/v1/forecast", 
                       params={"latitude": lat, "longitude": lon, "hourly": ["temperature_2m"], "forecast_days": 2}).json()
    df = pd.DataFrame(res["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    current = pd.Timestamp.now().tz_localize(None)
    print(df[df["time"] >= current].head(10))

def analyze_route(city_start: str, city_end: str, waypoints: int = 5):
    s_lat, s_lon = get_coordinates(city_start)
    e_lat, e_lon = get_coordinates(city_end)
    
    route_lats = np.linspace(s_lat, e_lat, waypoints)
    route_lons = np.linspace(s_lon, e_lon, waypoints)
    
    for lat, lon in zip(route_lats, route_lons):
        res = requests.get("https://api.open-meteo.com/v1/forecast", 
                           params={"latitude": lat, "longitude": lon, "current": ["temperature_2m"]}).json()
        print(f"Lat/Lon: {lat:.2f}, {lon:.2f} | Temp: {res.get('current', {}).get('temperature_2m')}°C")

if __name__ == "__main__":
    analyze_route("Hyderabad", "Mysuru", 14)

Lat/Lon: 17.38, 78.46 | Temp: 29.7°C
Lat/Lon: 16.99, 78.32 | Temp: 30.0°C
Lat/Lon: 16.60, 78.18 | Temp: 30.7°C
Lat/Lon: 16.21, 78.04 | Temp: 31.7°C
Lat/Lon: 15.82, 77.90 | Temp: 33.2°C
Lat/Lon: 15.43, 77.76 | Temp: 32.2°C
Lat/Lon: 15.04, 77.62 | Temp: 32.9°C
Lat/Lon: 14.65, 77.48 | Temp: 32.7°C
Lat/Lon: 14.25, 77.34 | Temp: 32.1°C
Lat/Lon: 13.86, 77.20 | Temp: 29.5°C
Lat/Lon: 13.47, 77.06 | Temp: 28.8°C
Lat/Lon: 13.08, 76.92 | Temp: 29.3°C
Lat/Lon: 12.69, 76.78 | Temp: 28.9°C
Lat/Lon: 12.30, 76.64 | Temp: 29.6°C


### Project Explanation
Translates human-readable cities into Lat/Lon coordinates via geocoding. Slices massive time-series JSONs into exactly the next 10 hours using Pandas datetime masking. Uses NumPy `linspace` to find mathematical geographical waypoints.

### Project Remarks
Linear interpolation calculates a straight line. Real production apps connect to Google Maps Directions API to fetch highway polylines.